# Optimal AC Policy

![](https://imgs.xkcd.com/comics/heat_pump.png)

## The Motivation

Though much less so now, historically I've been a bit of a meiser (I've been told) when it comes to AC usage in temperate Southern California. While I can accept that characterization, one accompanying assertion I've heard is that it would be "cheaper" to leave the AC unit on all the time instead of having "intervals" of having it turned on. Enough people have told me this (and I find it sufficiently ridiculous) for me to feel motivated to actually run some simulations to prove them definitively wrong. Perhaps a bit petty, but it's also fun to do some math.

## The Model

Internal **state** of an integrated AC system is appropriately the **indoor temperature** (parameterized in time):

$T_{\text{in}}(t)\in \mathbb{R}$,

with the **inputs** being the **outdoor temperature** and the AC unit **on/off state** (AC units only have two states: *full blast* and *off*):

$T_{\text{out}}(t)\in \mathbb{R}$, $AC(t) \in \{0,1\}$.

Indoor temperature reacts as a *first-order system* with respect to temperature difference and AC cooling:

$$C \cdot \frac{dT_{\text{in}}}{dt}=\frac{T_{\text{out}}(t)-T_{\text{in}}(t)}{R}-AC(t)\cdot \eta \cdot P_{\text{AC}},$$

where $C$ is the **thermal capacity** of the home ($J/^\circ C$), $R$ is the **thermal resistance** between inside and outside ($^\circ C/W$), and $\eta \cdot P_{\text{AC}}$ is the effective **cooling rate** of the AC when it is on ($W=J/s$). The cooling rate is equal to the power draw of the AC unit when it is on $P_{\text{AC}}$ scaled by a coefficient of performance factor $\eta$.

## The Controller

A standard AC unit will employ a kind of dead-simple **threshold-based control** with **hysteresis** to avoid switching on / off too rapidly when $T_{\text{in}}$ inevitably hovers around the desired value, wandering above or below that value in infinitesimal and rapid fashion. A key knob to turn in the simulation (and the source of the original argument) is the ability to turn the AC unit logic itself **on or off** with the variable $\Pi_{\text{AC}}\in\{0,1\}$. The pseudocode logic for this looks like:

- If $\Pi_{\text{AC}}=0$, then the AC is to be turned off. Disregard everything below.
- If $T_{\text{in}} > T_{\text{desired}} + \tau$ then turn on the AC.
- Else, if $T_{\text{in}} < T_{\text{desired}} - \tau$ then turn off the AC.
- Else, maintain previous on/off state.

The width of the temperature band is determined by the parameter $\tau$ ($^\circ C$).

## The Outputs

While one of the outputs (and decision points) in this simulation is obviously the cost of running the AC (which we calculate with the integral $\kappa=k\int P_{\text{AC}} AC(t)dt$ with electricity rate $k$ ($\$/kWh$)), the second output that we care about (to weigh against the cost) is the slightly more ambiguous / subjective measure of human comfort. I will define it thus:

- $\gamma$ - Dimensionless human "comfort score." A higher score means more comfort and is aggregated over discrete points in time.
- A temperature of $T_{\text{in}} \geq T_{\text{desired}} + \Gamma$ at any given moment corresponds to a comfort score of $\gamma=0$, where $\Gamma>\tau$ is a configurable parameter we can play with.
- $\gamma$ linearly scales to a value of $1$ as $T_{\text{in}}$ decreases to $T_{\text{in}} = T_{\text{desired}} + \tau$, at which point it maxes out.

So the overall **cost** associated with an AC policy would be some weighted sum of literal dollar cost and "discomfort" cost. We'll define it explicitly as:

$$W_\kappa \kappa^2 + W_\gamma \gamma^2,$$

where $W_\kappa$ and $W_\gamma$ are the relative (dimensionless) weights given to dollar versus discomfort cost, respectively.

## The Program

This being a simulation of many variables whose values may be uncertain and/or dependent on subjectivities, we're going to formulate our simulation as a function of a few different input parameters that we can tweak and run multiple simulations for:

- $T_{\text{in}}(0)$ - Initial interior temperature.
- $\tau$ - Band width of the AC controller.
- $\Gamma$ - Maximum discomfort temperature threshold.
- $W_\kappa$, $W_\gamma$ - Subjective measure comparing dollar cost to discomfort cost.

and hold all other variables constant (and representative for a town home in Southern California during "peak usage hours", which is also when it is hottest):

- $T_{\text{desired}}=$23C
- $T_{\text{out}}(t)=$27C
- $C\approx 5.5\times 10^6 J/^\circ C$
- $R\approx 5.5\times 10^{-2} ~^\circ C/W$
- $P_{\text{AC}}\approx$ 3,000 W
- $\eta \approx 3.5$
- $k\approx \$0.60/kWh$

The time interval will also be constant as $0\leq t \leq$ 60 minutes.

The actual **input** to the simulation (and thus **the thing that an optimizer is allowed to tune** to try to get an optimal outcome) will be $\Pi_{\text{AC}}(t)$, or the decision to turn the AC unit logic on or off at any given point in time.

The "optimal outcome" for the optimizer to figure out will be to minimize the cost function $W_\kappa \kappa^2 + W_\gamma \gamma^2$ summed over all time samples (let's sample at $\Delta t=$ 1-second time intervals).

## The Implementation